# M01 — Gravity station corrections

**Level 2 · Prerequisites: foundations notebook; read time: 25 minutes.**

## Objectives

This lesson executes the product station processor on its committed analytic control. It targets Bouguer disturbance, not density inversion. Gravity is downward-positive mGal; heights are upward WGS84 ellipsoidal metres. The authored control is not field data.

## Setup

Use the already reviewed Python 3.12 pipeline environment. This notebook installs nothing.

Import repository code and confirm its source file.

In [ ]:
import json, sys
from pathlib import Path
root=Path.cwd().parents[2]; sys.path.insert(0,str(root/'data-pipeline'))
from gravity_processing import process_survey
# Identify the implementation loaded from this repository.
print(Path(__import__('gravity_processing').__file__).resolve())

## Data

This lesson executes the product station processor on its committed analytic control. It targets Bouguer disturbance, not density inversion. Gravity is downward-positive mGal; heights are upward WGS84 ellipsoidal metres. The authored control is not field data.

**Files:** docs/methods/gravity-processing/examples/station-control.json

## Method

For latitude φ and receiver height h, D=g−γ(φ,h). The infinite land-plate attraction is B=2πGρt×10⁵ mGal, where G is m³ kg⁻¹ s⁻², ρ is kg m⁻³ and surface thickness t is m. Output D_B=D−B. Receiver height and terrain thickness are distinct; this is not automatic free-air or DEM terrain correction.

Let h be receiver ellipsoidal height and s the surface height used for the slab. The processor’s ordered terms are −γ(φ,0), γ(φ,0)−γ(φ,h), and −2πGρs. Their sum transforms absolute g into Bouguer disturbance. It stores the exact additions and hashes so the next run can reconstruct prior state and reject duplicate or reordered corrections. That state history is scientifically material: a second application would otherwise subtract normal gravity and the plate twice while still producing plausible-looking numbers.

For independent primitive errors, first-order propagation is σ²_out=∑ᵢ(∂y/∂xᵢ)²σ²ᵢ. The conservative marginal policy instead sums absolute contributions, ∑ᵢ|∂y/∂xᵢ|σᵢ; it does not require independence but is not a covariance model. Both are conditional on the supplied input sigmas. The implementation explicitly excludes regional terrain, uncertainty in the reference field, source bias and geological uncertainty. The one-station control can test units, correction ordering, deterministic history and refusal behavior, but cannot validate a regional map, outlier screen or field suitability.


Execute the product method on the declared control.

In [ ]:
request=json.loads((root/'docs/methods/gravity-processing/examples/station-control.json').read_text(encoding='utf-8'))
# Execute the product correction core on its committed analytic control.
result=process_survey(request['dataset'],request['config'])
print({'state':result['dataset']['state'],'derived_mGal':result['qc']['derived_mgal'],'uncertainty_mGal':result['processing']['uncertainty_mgal'],'accepted':result['processing']['full_method_accepted']})

Adversarial or negative control: isolate one assumption or violate one declared physical constraint.

In [ ]:
try:
    process_survey(result['dataset'],request['config'])
except ValueError as error:
    # Reapplying an already reached correction state must be refused.
    print('Expected duplicate-correction refusal:',str(error))
else:
    raise AssertionError('Repeated correction was accepted')

The USGS manual describes upstream acquisition and QA ([TM 2-D4](https://pubs.usgs.gov/tm/02/d04/tm2d4.pdf)); Harmonica documents plate attraction ([API reference](https://www.fatiando.org/harmonica/latest/api/index.html)). Conditional uncertainty propagates declared marginal errors; it is not geological confidence. One analytic station cannot evaluate regional maps, terrain or field eligibility.

The plotting cell reuses the actual `process_survey` result and reruns only with the other declared error aggregation. All bars are mGal; the chart is a one-station analytic control, not a spatial correction map.


In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt

# Decompose the actual pipeline history and compare its two implemented uncertainty policies.
h = result['dataset']['history']; g = request['dataset']['stations'][0]['value_mgal']
terms = [g + h[0]['additions_mgal'][0], h[1]['additions_mgal'][0], h[2]['additions_mgal'][0]]
cumulative = np.cumsum(terms)
other = copy.deepcopy(request['config']); other['uncertainty_model'] = 'independent_first_order'
independent = process_survey(request['dataset'], other)
fig,(ax,bx)=plt.subplots(1,2,figsize=(11,4.8),constrained_layout=True)
labels=['Reference: g−γ₀','Height: γ₀−γh','Plate: −B']
ax.bar(labels,terms,color=['#4477aa','#66ccee','#cc6677']); ax.plot(range(3),cumulative,'ko--',label='Cumulative disturbance')
ax.axhline(0,color='.4',lw=.8); ax.set(ylabel='Gravity contribution [mGal]',title='Ordered M01 transform'); ax.tick_params(axis='x',rotation=15); ax.legend()
bx.bar(['Independent RSS','Conservative marginal sum'],[independent['processing']['uncertainty_mgal'][0],result['processing']['uncertainty_mgal'][0]],color=['#228833','#aa3377'])
bx.set(ylabel='Propagated uncertainty [mGal]',title='Same input errors, different dependence policy')
print('recorded history:',[x['name'] for x in h]); print('final state:',result['dataset']['state'],'accepted:',result['processing']['full_method_accepted'])
plt.show()


## Results

Expect about 12 mGal, an ordered correction history and `full_method_accepted=false`. Reapplication must fail. This is local arithmetic/state evidence, not full M01 acceptance.

## Summary

A correction is a typed ordered transformation. Preserve units, reference, original value and history; never infer missing metadata from numeric magnitude.




## References

- https://pubs.usgs.gov/tm/02/d04/tm2d4.pdf
- https://www.fatiando.org/harmonica/latest/api/index.html